# 0. AMI data preparation and aggregation to transformer nodes

**Purpose:** load the individual AMI file, keep 2024, average duplicate (service point, timestamp) rows, drop Feb 29 (the ResStock models have no leap day), and write the 15-min and hourly AMI files to `data/parquet/`. Then aggregate the meters to transformer nodes (`<xfr_gisid>_RES` / `<xfr_gisid>_NRS`), with a choice of how to handle AMI gaps.

**Position in the pipeline:** first step; independent of the ResStock notebooks. Its outputs feed the external matching step (`ami_ind_data_15`) and notebook 4 (`ami_meter_xfr_map.csv`, `ami_meter_wide_15_<gap_tag>.parquet`). See `docs/pipeline_summary.md`.

| # | Section | Key output |
|---|---------|-----------|
| 1 | Prepare AMI data | `data/parquet/ami_ind_data_15.*`, `ami_ind_data_hour.*` |
| 2 | Duplicate / multi-channel diagnostic (optional) | `data/parquet/ami_duplicates_2024.csv` |
| 3 | Aggregate AMI to transformer nodes | `results/aggregated_load_nodes_wide_ami_<gap_tag>.*`, `results/aggregated_load_nodes_ami_<gap_tag>_report.csv`, `results/ami_meter_xfr_map.csv`, `results/ami_meter_gap_summary.csv`, `data/parquet/ami_meter_wide_15_<gap_tag>.parquet` |

**Inputs**
- `data/input_files/consumption/ch1_4301002_individual.csv`: raw AMI (`SERVICEPOINTID`, `CHANNELNUMBER`, `ENDTIME_EST`, `INTERVAL_READ`).
- `data/input_files/location_reference.csv`: service point → transformer (`servicepointid`, `xfr_gisid`, `Residential`).

**Notes**
- Duplicate timestamps all come from the daylight-saving fall-back hour (Nov 3, 01:00–01:45); they are averaged. The AMI `ENDTIME_EST` is therefore local clock time, not strict EST (the spring-forward hour Mar 10 02:00–02:45 is missing for every meter).
- Timestamps are interval-end; values are kWh per 15 min. The hourly table groups each 15-min value by `floor(hour)` of its timestamp, the same convention as the ResStock hourly tables (notebooks 1 and 3).
- Supersedes the AMI part of `Aggregate AMI Load to Feeder Node.ipynb`.


## Setup
Assumes the project environment is already created and activated (`python -m pip install -r requirements.txt`). The two setup cells below are the same in every notebook of this project; run them first. Run notebooks from `grid_edge_fy25/notebooks/` (all paths are relative to it).

In [1]:
# Preflight check: assumes the environment was created from requirements.txt
import importlib.util

required_modules = ['pandas', 'numpy', 'pyarrow']
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
if missing:
    raise RuntimeError(
        "Missing Python packages: " + ", ".join(missing) +
        "\nCreate/activate the project environment and run: python -m pip install -r requirements.txt"
    )
print("Preflight OK: environment is ready.")


Preflight OK: environment is ready.


In [2]:
# Common setup (identical in all notebooks of this project)
import ast
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

# Paths (relative to grid_edge_fy25/notebooks/)
current_path = Path(os.getcwd())
data_path = current_path / ".." / "data"
base_files = data_path / "base_files"        # building id lists, measures, weather, workflow
input_files = data_path / "input_files"      # NY metadata, AMI consumption
parquet_folder = data_path / "parquet"       # load tables written for the optimization
results_path = current_path / ".." / "results"
extract_dir = data_path / "resstock_models"  # bldgXXXXXXX folders (models + simulation output)

parquet_folder.mkdir(parents=True, exist_ok=True)


---
## 1. Prepare AMI data
Loads the individual AMI file (`AMI_FILE`), keeps 2024, averages duplicate (service point, timestamp) rows, drops Feb 29, and writes 15-min and hourly files to `data/parquet/`. Edit `AMI_FILE` if the raw file is elsewhere.


In [ ]:
# Raw AMI: SERVICEPOINTID, CHANNELNUMBER, ENDTIME_EST (interval end), INTERVAL_READ (kWh per 15 min)
AMI_FILE = input_files / "consumption" / "ch1_4301002_individual.csv"
KEYS = ["SERVICEPOINTID", "ENDTIME_EST"]

df = pd.read_csv(AMI_FILE)
df["ENDTIME_EST"] = pd.to_datetime(df["ENDTIME_EST"])
df_2024_raw = df[df["ENDTIME_EST"].dt.year == 2024].copy()   # un-averaged rows (keeps CHANNELNUMBER); used in section 2
del df
print("2024 rows:", len(df_2024_raw), "| service points:", df_2024_raw["SERVICEPOINTID"].nunique())

# Average duplicate (service point, timestamp) rows; single rows are unchanged
df_2024 = df_2024_raw.groupby(KEYS, as_index=False)["INTERVAL_READ"].mean()

# Drop Feb 29 (the ResStock models run on a 365-day calendar)
df_2024 = df_2024[df_2024["ENDTIME_EST"].dt.strftime("%m-%d") != "02-29"]

# Hourly sums per service point (resample creates empty Feb 29 bins, dropped again)
df_2024_hour = (
    df_2024.set_index("ENDTIME_EST")
           .groupby("SERVICEPOINTID")["INTERVAL_READ"]
           .resample("h").sum()
           .reset_index()
)
df_2024_hour = df_2024_hour[df_2024_hour["ENDTIME_EST"].dt.strftime("%m-%d") != "02-29"]

df_2024.to_parquet(parquet_folder / "ami_ind_data_15.parquet", index=False)
df_2024.to_csv(parquet_folder / "ami_ind_data_15.csv", index=False)
df_2024_hour.to_parquet(parquet_folder / "ami_ind_data_hour.parquet", index=False)
df_2024_hour.to_csv(parquet_folder / "ami_ind_data_hour.csv", index=False)

print("15-min rows:", len(df_2024), "| hourly rows:", len(df_2024_hour))
print("Feb 29 left? 15-min:", (df_2024["ENDTIME_EST"].dt.strftime("%m-%d") == "02-29").any(),
      "| hourly:", (df_2024_hour["ENDTIME_EST"].dt.strftime("%m-%d") == "02-29").any())
print("Saved:", parquet_folder / "ami_ind_data_15.parquet", "and", parquet_folder / "ami_ind_data_hour.parquet")
df_2024.head()


Total rows: 21380228
Total unique service IDs: 670
15-min has Feb 29?  False
Hourly has Feb 29?  False
Saved: ..\data\parquet\ami_ind_data_15.parquet
   SERVICEPOINTID         ENDTIME_EST  INTERVAL_READ
0      6000973858 2024-01-01 00:00:00          0.095
1      6000973858 2024-01-01 00:15:00          0.112
2      6000973858 2024-01-01 00:30:00          0.118
3      6000973858 2024-01-01 00:45:00          0.069
4      6000973858 2024-01-01 01:00:00          0.081


---
## 2. Check duplicates (optional)
Diagnostic: counts service points with more than one channel and the duplicate (service point, timestamp) rows, and saves the duplicates to `data/parquet/ami_duplicates_2024.csv`. Needs `df_2024_raw` from section 1 (the un-averaged 2024 rows).


In [ ]:
# Service points with more than one channel (separate timeseries)
n_channels = df_2024_raw.groupby("SERVICEPOINTID")["CHANNELNUMBER"].nunique()
print("Service points with >1 channel:", int((n_channels > 1).sum()))

# Duplicate (service point, timestamp) rows
df_dupes = df_2024_raw[df_2024_raw.duplicated(subset=KEYS, keep=False)]
print("Duplicate rows:", len(df_dupes), "| service points with duplicates:", df_dupes["SERVICEPOINTID"].nunique())
print("Duplicate timestamps:", sorted(df_dupes["ENDTIME_EST"].dt.strftime("%Y-%m-%d %H:%M").unique())[:8])

df_dupes.to_csv(parquet_folder / "ami_duplicates_2024.csv", index=False)
print("Saved:", parquet_folder / "ami_duplicates_2024.csv")

# Section 1 averaged these rows: one row per unique (service point, timestamp)
print("Raw 2024 rows:", len(df_2024_raw), "| unique (service point, timestamp):", len(df_2024_raw.drop_duplicates(KEYS)))


Number of service IDs with >1 timeseries: 0
Number of duplicate rows: 5358
Number of service IDs with duplicate rows: 670
Saved duplicates to: ..\data\parquet\ami_duplicates_2024.csv
Before rows: 21380228
After rows : 21377549
   SERVICEPOINTID         ENDTIME_EST  INTERVAL_READ_avg
0      6000973858 2024-01-01 00:00:00              0.095
1      6000973858 2024-01-01 00:15:00              0.112
2      6000973858 2024-01-01 00:30:00              0.118
3      6000973858 2024-01-01 00:45:00              0.069
4      6000973858 2024-01-01 01:00:00              0.081


---
## 3. Aggregate AMI to transformer nodes
Independent of sections 1–2 at run time: it reads `data/parquet/ami_ind_data_15.parquet` and `data/input_files/location_reference.csv`.

**Choices**
- **AMI gaps** (`GAP_METHOD`):
  - `"zero"`: missing intervals count as 0 kWh (behavior of the original notebook).
  - `"nan"`: missing intervals stay missing; a transformer column is NaN at any interval where one of its meters is missing.
  - `"interpolate"`: linear interpolation of interior gaps of at most `INTERP_MAX_GAP` intervals; longer gaps and gaps at the start/end of a meter's record stay NaN.
- **Meters on several transformers** (62 in the current file): the load is split equally (`weight = 1 / n_xfr`), so meter energy is conserved.
- **Sector**: `Residential == "Res"` → `_RES`, otherwise `_NRS`.

The output file names carry the gap tag (`zero`, `nan`, `interp<N>`), so several variants can coexist.

### 3.1 Parameters and meter-wide AMI table

In [4]:
# ---- Parameters ----
GAP_METHOD = "zero"     # "zero" | "nan" | "interpolate"
INTERP_MAX_GAP = 4      # "interpolate" only: longest interior gap filled, in 15-min intervals (4 = 1 h)
LOC_FILE = input_files / "location_reference.csv"

if GAP_METHOD not in {"zero", "nan", "interpolate"}:
    raise ValueError(f"Unknown GAP_METHOD: {GAP_METHOD}")
GAP_TAG = {"zero": "zero", "nan": "nan", "interpolate": f"interp{INTERP_MAX_GAP}"}[GAP_METHOD]
print("Gap method:", GAP_METHOD, "| tag:", GAP_TAG)

# Full 2024 15-min grid (interval-end), Feb 29 removed -> 35,040 timestamps
time_index = pd.date_range("2024-01-01 00:00", "2024-12-31 23:45", freq="15min", name="ENDTIME_EST")
time_index = time_index[time_index.strftime("%m-%d") != "02-29"]

# Meter-wide AMI table (time x service point), before gap handling
ami_15 = pd.read_parquet(parquet_folder / "ami_ind_data_15.parquet")
ami_15["SERVICEPOINTID"] = ami_15["SERVICEPOINTID"].astype(str)
ami_wide_raw = (
    ami_15.pivot(index="ENDTIME_EST", columns="SERVICEPOINTID", values="INTERVAL_READ")
          .reindex(time_index)
)
ami_wide_raw.columns.name = None
del ami_15
print("Meter-wide AMI:", ami_wide_raw.shape)

all_missing = ami_wide_raw.index[ami_wide_raw.isna().all(axis=1)]
print("Timestamps missing for every meter:", len(all_missing))

Gap method: zero | tag: zero
Meter-wide AMI: (35040, 670)
Timestamps missing for every meter: 515


### 3.2 Meter gap summary and gap handling
Writes a per-meter gap summary (`results/ami_meter_gap_summary.csv`: missing intervals, coverage, longest gap, first/last reading), then applies `GAP_METHOD`. The gap-handled meter-wide table is saved to `data/parquet/ami_meter_wide_15_<gap_tag>.parquet` (used by notebook 4 for the AMI fallback of unmatched residential meters).


In [5]:
# Per-meter gap runs: used for the summary and for the "interpolate" mask
isna = ami_wide_raw.isna().to_numpy()
short_gap = np.zeros_like(isna)   # True where a NaN belongs to an interior gap <= INTERP_MAX_GAP
gap_rows = []
for j, spid in enumerate(ami_wide_raw.columns):
    m = isna[:, j]
    edges = np.diff(np.concatenate(([0], m.astype(np.int8), [0])))
    starts, ends = np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)
    lengths = ends - starts
    for s, e in zip(starts, ends):
        interior = s > 0 and e < len(m)
        if interior and (e - s) <= INTERP_MAX_GAP:
            short_gap[s:e, j] = True
    valid = ami_wide_raw.index[~m]
    gap_rows.append({
        "servicepointid": spid,
        "n_missing": int(m.sum()),
        "coverage_pct": round(100 * (1 - m.mean()), 2),
        "n_gaps": len(lengths),
        "longest_gap_intervals": int(lengths.max()) if len(lengths) else 0,
        "first_reading": valid.min() if len(valid) else pd.NaT,
        "last_reading": valid.max() if len(valid) else pd.NaT,
    })
gap_summary = pd.DataFrame(gap_rows)
gap_summary.to_csv(results_path / "ami_meter_gap_summary.csv", index=False)
print("Meters:", len(gap_summary),
      "| median coverage %:", gap_summary["coverage_pct"].median(),
      "| meters < 95% coverage:", int((gap_summary["coverage_pct"] < 95).sum()))

# Apply the gap method
if GAP_METHOD == "zero":
    ami_wide = ami_wide_raw.fillna(0.0)
elif GAP_METHOD == "nan":
    ami_wide = ami_wide_raw.copy()
else:  # interpolate: fill only short interior gaps, longer ones stay NaN
    interp_all = ami_wide_raw.interpolate(method="linear", limit_area="inside")
    ami_wide = ami_wide_raw.mask(short_gap, interp_all)
    print("Intervals filled by interpolation:", int(short_gap.sum()))

print("Remaining NaN cells:", int(ami_wide.isna().sum().sum()))
ami_wide.to_parquet(parquet_folder / f"ami_meter_wide_15_{GAP_TAG}.parquet")
print("Saved:", parquet_folder / f"ami_meter_wide_15_{GAP_TAG}.parquet")

Meters: 670 | median coverage %: 92.18 | meters < 95% coverage: 638
Remaining NaN cells: 0
Saved: ..\data\parquet\ami_meter_wide_15_zero.parquet


### 3.3 Meter → transformer map
One row per (service point, transformer). `n_xfr` = number of transformers the meter is listed on, `weight = 1 / n_xfr`, `column` = output column name, `has_ami` = meter has 2024 AMI data. Saved to `results/ami_meter_xfr_map.csv`. Meters without a transformer and location-file meters without AMI are reported.

In [6]:
loc = pd.read_csv(LOC_FILE, dtype={"servicepointid": str})
loc["servicepointid"] = loc["servicepointid"].str.strip()

meter_xfr = (
    loc.dropna(subset=["xfr_gisid"])
       .assign(
           xfr_gisid=lambda d: d["xfr_gisid"].astype("int64").astype(str),
           sector=lambda d: np.where(
               d["Residential"].astype(str).str.strip().str.lower().str.startswith("res"), "RES", "NRS"),
       )
       .drop_duplicates(["servicepointid", "xfr_gisid"])
       [["servicepointid", "xfr_gisid", "sector"]]
       .reset_index(drop=True)
)
inconsistent_sector = meter_xfr.groupby("servicepointid")["sector"].nunique().loc[lambda s: s > 1]
if len(inconsistent_sector):
    print("WARNING: meters with both RES and NRS rows:", list(inconsistent_sector.index))

meter_xfr["n_xfr"] = meter_xfr.groupby("servicepointid")["xfr_gisid"].transform("nunique")
meter_xfr["weight"] = 1.0 / meter_xfr["n_xfr"]
meter_xfr["column"] = meter_xfr["xfr_gisid"] + "_" + meter_xfr["sector"]
meter_xfr["has_ami"] = meter_xfr["servicepointid"].isin(ami_wide.columns)
meter_xfr.to_csv(results_path / "ami_meter_xfr_map.csv", index=False)

loc_spids = set(loc["servicepointid"])
no_xfr = sorted(loc_spids - set(meter_xfr["servicepointid"]))
no_ami = sorted(loc_spids - set(ami_wide.columns))
ami_not_in_loc = sorted(set(ami_wide.columns) - loc_spids)

print("Location-file meters:", len(loc_spids), "| transformers:", meter_xfr["xfr_gisid"].nunique())
print("Meters on >1 transformer (load split):", meter_xfr.loc[meter_xfr["n_xfr"] > 1, "servicepointid"].nunique())
print("Meters without transformer (excluded):", no_xfr)
print("Location-file meters without 2024 AMI:", no_ami)
print("AMI meters not in location file (excluded):", ami_not_in_loc)
print("Saved:", results_path / "ami_meter_xfr_map.csv")

Location-file meters: 676 | transformers: 177
Meters on >1 transformer (load split): 62
Meters without transformer (excluded): ['6003668337']
Location-file meters without 2024 AMI: ['6000979540', '6000983426', '6000989800', '6000991593', '6000993734', '6000999010']
AMI meters not in location file (excluded): []
Saved: c:\Users\jxiong\Documents\GitHub\grid_edge_ithaca\grid_edge_fy25\notebooks\..\results\ami_meter_xfr_map.csv


### 3.4 Aggregate and save
Each column `<xfr_gisid>_<RES|NRS>` = Σ weight × meter load over the meters with AMI on that transformer and sector. Missing values propagate (`skipna=False`), so with `"nan"` / `"interpolate"` a column is NaN wherever one of its meters is still missing. Writes `results/aggregated_load_nodes_wide_ami_<gap_tag>.csv` / `.parquet` (index `ENDTIME_EST`, kWh per 15 min) and a per-column report `results/aggregated_load_nodes_ami_<gap_tag>_report.csv`.

In [7]:
mapped = meter_xfr[meter_xfr["has_ami"]]
agg_cols = {}
for col, g in mapped.groupby("column", sort=True):
    agg_cols[col] = (
        ami_wide[g["servicepointid"].tolist()]
        .mul(g["weight"].to_numpy(), axis=1)
        .sum(axis=1, skipna=False)
    )
agg_ami = pd.DataFrame(agg_cols, index=ami_wide.index)

# Per-column report (all transformer/sector columns in the location file, including those without AMI)
report = (
    meter_xfr.groupby("column")
    .agg(xfr_gisid=("xfr_gisid", "first"), sector=("sector", "first"),
         n_meters=("servicepointid", "size"),
         n_meters_ami=("has_ami", "sum"),
         n_meters_split=("n_xfr", lambda s: int((s > 1).sum())))
)
report["annual_kWh"] = agg_ami.sum().reindex(report.index)          # NaN intervals skipped
report["nan_intervals"] = agg_ami.isna().sum().reindex(report.index)
report["in_output"] = report.index.isin(agg_ami.columns)
report = report.reset_index()

out_stem = results_path / f"aggregated_load_nodes_wide_ami_{GAP_TAG}"
agg_ami.to_csv(out_stem.with_suffix(".csv"))
agg_ami.to_parquet(out_stem.with_suffix(".parquet"))
report.to_csv(results_path / f"aggregated_load_nodes_ami_{GAP_TAG}_report.csv", index=False)

n_res = sum(c.endswith("_RES") for c in agg_ami.columns)
print(f"Columns: {agg_ami.shape[1]} ({n_res} RES, {agg_ami.shape[1] - n_res} NRS) x {len(agg_ami)} timestamps")
print("Columns without any AMI meter (not in output):", report.loc[~report["in_output"], "column"].tolist())
print("Saved:", out_stem.with_suffix(".csv"))

Columns: 215 (145 RES, 70 NRS) x 35040 timestamps
Columns without any AMI meter (not in output): ['220269836_RES', '220270701_NRS']
Saved: c:\Users\jxiong\Documents\GitHub\grid_edge_ithaca\grid_edge_fy25\notebooks\..\results\aggregated_load_nodes_wide_ami_zero.csv


### 3.5 Checks
Energy conservation: the split weights of each meter sum to 1, so total meter energy must equal total column energy. With `"zero"` this is exact; with `"nan"` / `"interpolate"` the difference is the energy at intervals where a column is NaN because another meter on it is missing.

In [8]:
mapped_spids = mapped["servicepointid"].unique().tolist()
meter_kwh = ami_wide[mapped_spids].sum().sum()   # NaN skipped
agg_kwh = agg_ami.sum().sum()                    # NaN skipped
print(f"Mapped meter energy : {meter_kwh:,.1f} kWh")
print(f"Aggregated energy   : {agg_kwh:,.1f} kWh")
print(f"Difference          : {meter_kwh - agg_kwh:,.1f} kWh")
if GAP_METHOD == "zero":
    assert np.isclose(meter_kwh, agg_kwh, rtol=1e-9), "Energy not conserved"
    print("Energy conserved.")
else:
    print("Columns with NaN intervals:", int((agg_ami.isna().sum() > 0).sum()),
          "| max NaN intervals in a column:", int(agg_ami.isna().sum().max()))

report.sort_values("annual_kWh", ascending=False).head(10)

Mapped meter energy : 6,893,245.7 kWh
Aggregated energy   : 6,893,245.7 kWh
Difference          : 0.0 kWh
Energy conserved.


,column,xfr_gisid,sector,n_meters,n_meters_ami,n_meters_split,annual_kWh,nan_intervals,in_output
190,220529264_NRS,220529264,NRS,2,2,0,561907.3200,0.0,True
205,220614028_NRS,220614028,NRS,2,2,0,482834.9460,0.0,True
165,220270665_NRS,220270665,NRS,1,1,0,249504.9200,0.0,True
85,220270522_NRS,220270522,NRS,26,26,0,225057.7555,0.0,True
119,220270568_RES,220270568,RES,22,22,0,125956.9620,0.0,True
68,220270493_RES,220270493,RES,11,11,0,112725.0225,0.0,True
69,220270494_RES,220270494,RES,8,8,0,103628.3670,0.0,True
75,220270499_RES,220270499,RES,9,9,0,101897.5770,0.0,True
92,220270544_RES,220270544,RES,9,9,0,95695.7805,0.0,True
149,220270647_RES,220270647,RES,15,15,0,89312.6070,0.0,True
